In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from pyod.models.hbos import HBOS
from pyod.models.copod import COPOD

import torch

In [2]:
f_data = pd.read_excel(r"C:\Users\royr\Downloads\Sample_fraud_analytics_intern_v1.1.xlsx")

In [3]:
# --- Missing values check ---
missing_values = f_data.isnull().sum()

print("Missing values per column:\n", missing_values)

Missing values per column:
 timestamp         0
date              0
year              0
month             0
month_start       0
day               0
day_of_week       0
hour              0
month_index       0
vendor_id         0
category          0
region            0
quantity          0
unit_price        0
total_price       0
po_number         0
invoice_number    0
dtype: int64


In [4]:
f_data['quantity'] = pd.to_numeric(f_data['quantity'], errors='coerce')
f_data['unit_price'] = pd.to_numeric(f_data['unit_price'], errors='coerce')
f_data['total_price'] = pd.to_numeric(f_data['total_price'], errors='coerce')

# Record rows with missing amount fields before any later imputation or fill steps.
f_data['has_missing_amount_fields'] = f_data[['quantity', 'unit_price', 'total_price']].isna().any(axis=1)

# Impute algebraically only when exactly one of the three amount fields is missing.
amount_cols = ['quantity', 'unit_price', 'total_price']
missing_count = f_data[amount_cols].isna().sum(axis=1)

mask_quantity_missing = (
    (missing_count == 1)
    & f_data['quantity'].isna()
    & f_data['unit_price'].notna()
    & f_data['total_price'].notna()
)
mask_unit_price_missing = (
    (missing_count == 1)
    & f_data['unit_price'].isna()
    & f_data['quantity'].notna()
    & f_data['total_price'].notna()
)
mask_total_price_missing = (
    (missing_count == 1)
    & f_data['total_price'].isna()
    & f_data['quantity'].notna()
    & f_data['unit_price'].notna()
)

f_data.loc[mask_quantity_missing, 'quantity'] = (
    f_data.loc[mask_quantity_missing, 'total_price'] / f_data.loc[mask_quantity_missing, 'unit_price']
)
f_data.loc[mask_unit_price_missing, 'unit_price'] = (
    f_data.loc[mask_unit_price_missing, 'total_price'] / f_data.loc[mask_unit_price_missing, 'quantity']
)
f_data.loc[mask_total_price_missing, 'total_price'] = (
    f_data.loc[mask_total_price_missing, 'quantity'] * f_data.loc[mask_total_price_missing, 'unit_price']
)

print("Rows with missing amount fields:", f_data['has_missing_amount_fields'].sum())
print("Rows with exactly one amount field missing:", int((missing_count == 1).sum()))

Rows with missing amount fields: 20
Rows with exactly one amount field missing: 5


In [5]:
# Hour cyclical encoding (0–23)
f_data['hour_sin'] = np.sin(2 * np.pi * f_data['hour'] / 24)
f_data['hour_cos'] = np.cos(2 * np.pi * f_data['hour'] / 24)

# Day of week cyclical encoding (0–6)
f_data['dow_sin'] = np.sin(2 * np.pi * f_data['day_of_week'] / 7)
f_data['dow_cos'] = np.cos(2 * np.pi * f_data['day_of_week'] / 7)

# Check results
print(f_data[['hour','hour_sin','hour_cos','day_of_week','dow_sin','dow_cos']].head())


   hour  hour_sin      hour_cos  day_of_week   dow_sin   dow_cos
0    23 -0.258819  9.659258e-01            6 -0.781831  0.623490
1    16 -0.866025 -5.000000e-01            3  0.433884 -0.900969
2    18 -1.000000 -1.836970e-16            5 -0.974928 -0.222521
3     5  0.965926  2.588190e-01            6 -0.781831  0.623490
4    11  0.258819 -9.659258e-01            4 -0.433884 -0.900969


In [6]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(f_data[['quantity','unit_price','total_price','hour_sin','hour_cos','dow_sin','dow_cos']])

In [7]:
# Flag exact duplicate invoice submissions based on the line-item signature.
duplicate_key_cols = ['vendor_id', 'category', 'region', 'quantity', 'unit_price', 'total_price', 'po_number', 'invoice_number']
f_data['is_duplicate_submission'] = f_data.duplicated(subset=duplicate_key_cols, keep=False)

print("Rows in duplicate submission groups:", f_data['is_duplicate_submission'].sum())
print(
    "Duplicate submission groups:",
    f_data.loc[f_data['is_duplicate_submission']].groupby(duplicate_key_cols).size().reset_index(name='row_count').shape[0]
)

Rows in duplicate submission groups: 1518
Duplicate submission groups: 697


In [8]:
expected_total = (f_data['quantity'] * f_data['unit_price']).round(2)
stated_total = f_data['total_price'].round(2)

valid_mask = f_data[['quantity', 'unit_price', 'total_price']].notna().all(axis=1)
disparity = (expected_total - stated_total).abs()

f_data['price_mismatch_flag'] = False
f_data.loc[valid_mask, 'price_mismatch_flag'] = (disparity.loc[valid_mask] > 0.02)

mismatch_rows = f_data.loc[f_data['price_mismatch_flag'], ['quantity', 'unit_price', 'total_price', 'price_mismatch_flag']]

print(f"Rows with mismatched totals: {mismatch_rows.shape[0]}")
mismatch_rows.head()

Rows with mismatched totals: 54


,quantity,unit_price,total_price,price_mismatch_flag
200,32.0,0.00,252218.56,True
255,0.0,0.00,465.70,True
8308,0.0,0.00,385.74,True
8340,0.0,0.00,179.36,True
17753,697.0,0.16,2113.66,True


In [9]:
f_data['total'] = f_data['quantity'] * f_data['unit_price']

In [10]:
key_cols = ['invoice_number', 'po_number', 'timestamp']
amount_cols = ['quantity', 'unit_price', 'total_price', 'total']

amount_variation = f_data.groupby(key_cols)[amount_cols].nunique()
mismatch_groups = amount_variation[(amount_variation > 1).any(axis=1)]

f_data['same_invoice_po_timestamp_diff_amount_flag'] = (
    f_data.set_index(key_cols).index.isin(mismatch_groups.index)
)

print("Rows flagged:", f_data['same_invoice_po_timestamp_diff_amount_flag'].sum())
f_data.loc[
    f_data['same_invoice_po_timestamp_diff_amount_flag'],
    key_cols + amount_cols
].head(10)

Rows flagged: 12


,invoice_number,po_number,timestamp,quantity,unit_price,total_price,total
199,INV678084,PO811407,2021-08-16 19:14:57,0.000000,0.000000,0.00,0.000
200,INV781696,PO508392,2021-09-09 10:51:25,32.000000,0.000000,252218.56,0.000
203,INV572177,PO874788,2021-11-03 00:41:48,312.019231,0.104000,32.45,32.450
255,INV547541,PO509531,2022-04-08 02:43:05,0.000000,0.000000,465.70,0.000
273,INV686122,PO436251,2022-01-20 01:13:24,7.000000,10789.930000,75529.51,75529.510
278,INV956510,PO271307,2022-01-16 05:00:37,294.000000,0.144014,42.34,42.340
17203,INV678084,PO811407,2021-08-16 19:14:57,308.000000,0.170000,52.36,52.360
17204,INV781696,PO508392,2021-09-09 10:51:25,32.000000,7881.830000,252218.56,252218.560
17207,INV572177,PO874788,2021-11-03 00:41:48,312.000000,0.104000,32.45,32.448
17259,INV547541,PO509531,2022-04-08 02:43:05,2.000000,232.850000,465.70,465.700


In [11]:
f_data['total_vs_total_price_diff'] = (f_data['total'] - f_data['total_price']).abs()
f_data['total_vs_total_price_flag'] = f_data['total_vs_total_price_diff'] > 0.01

print("Rows with total mismatch:", f_data['total_vs_total_price_flag'].sum())
f_data.loc[
    f_data['total_vs_total_price_flag'],
    ['quantity', 'unit_price', 'total', 'total_price', 'total_vs_total_price_diff']
].head()

Rows with total mismatch: 54


,quantity,unit_price,total,total_price,total_vs_total_price_diff
200,32.0,0.00,0.00,252218.56,252218.56
255,0.0,0.00,0.00,465.70,465.70
8308,0.0,0.00,0.00,385.74,385.74
8340,0.0,0.00,0.00,179.36,179.36
17753,697.0,0.16,111.52,2113.66,2002.14


In [12]:
f_data['avg_quantity_vendor_item'] = (
    f_data.groupby(['vendor_id', 'category'])['quantity']
    .transform('mean')
)

f_data['std_quantity_vendor_item'] = (
    f_data.groupby(['vendor_id', 'category'])['quantity']
    .transform('std')
)


In [13]:
f_data['std_quantity_vendor_item'] = f_data['std_quantity_vendor_item'].fillna(1).replace(0, 1)
f_data['item_quantity_zscore'] = np.where(
    f_data['quantity'].notna() & f_data['avg_quantity_vendor_item'].notna() & f_data['std_quantity_vendor_item'].notna(),
    (f_data['quantity'] - f_data['avg_quantity_vendor_item']) / f_data['std_quantity_vendor_item'],
    0.0
)

In [14]:
vendor_region_mode = (
    f_data.groupby('vendor_id')['region']
    .agg(lambda x: x.mode()[0] if not x.mode().empty else None)
    .reset_index(name='vendor_region_mode')
)

# Merge back
f_data = f_data.merge(vendor_region_mode, on='vendor_id', how='left')

# Flag region change
f_data['region_changed'] = f_data['region'] != f_data['vendor_region_mode']

In [15]:
# Vendor's most common transaction hour
f_data['transaction_hour'] = f_data['timestamp'].dt.hour

vendor_hour_mode = (
    f_data.groupby('vendor_id')['transaction_hour']
    .agg(lambda x: x.mode()[0] if not x.mode().empty else None)
    .reset_index(name='vendor_hour_mode')
)

f_data = f_data.merge(vendor_hour_mode, on='vendor_id', how='left')

# Flag unusual hours
f_data['is_outside_usual_hours'] = f_data['transaction_hour'] != f_data['vendor_hour_mode']

In [16]:
# Sort by vendor and timestamp
f_data = f_data.sort_values(['vendor_id', 'timestamp'])

# Time difference between consecutive transactions
f_data['inter_transaction_time'] = f_data.groupby('vendor_id')['timestamp'].diff().dt.total_seconds()

# Flag burst activity (e.g., < 300 seconds = 5 minutes)
f_data['is_burst_activity'] = f_data['inter_transaction_time'] < 300

In [17]:
# Vendor historical mean and std of amounts
f_data['vendor_amount_mean'] = f_data.groupby('vendor_id')['total_price'].transform('mean')
f_data['vendor_amount_std'] = f_data.groupby('vendor_id')['total_price'].transform('std').fillna(1).replace(0, 1)

# Z-score for transaction amount, with a safe fallback so missing input does not create NaN values.
f_data['amount_zscore'] = np.where(
    f_data['total_price'].notna() & f_data['vendor_amount_mean'].notna() & f_data['vendor_amount_std'].notna(),
    (f_data['total_price'] - f_data['vendor_amount_mean']) / f_data['vendor_amount_std'],
    0.0
)

# Flag outliers (e.g., > 3 std deviations)
f_data['is_amount_outlier'] = f_data['amount_zscore'].abs() > 3

In [18]:
# Step 1: Create a proper date column from year, month, day
f_data['date'] = pd.to_datetime(f_data[['year','month','day']])

# Step 2: Count transactions per vendor per day
daily_counts = (
    f_data.groupby(['vendor_id','date'])
    .size()
    .reset_index(name='daily_txn_count')
)

# Step 3: Compute rolling 7-day average per vendor
daily_counts['rolling_txn_count'] = (
    daily_counts.groupby('vendor_id')['daily_txn_count']
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

# Step 4: Merge back into main DataFrame
f_data = f_data.merge(daily_counts, on=['vendor_id','date'], how='left')

# Step 5: Compare rolling count to historical mean
f_data['vendor_txn_mean'] = f_data.groupby('vendor_id')['daily_txn_count'].transform('mean')
f_data['frequency_change_ratio'] = np.where(
    f_data['rolling_txn_count'].notna() & f_data['vendor_txn_mean'].notna() & (f_data['vendor_txn_mean'] != 0),
    f_data['rolling_txn_count'] / f_data['vendor_txn_mean'],
    0.0
)

# Flag spikes/drops
f_data['is_frequency_spike_drop'] = (f_data['frequency_change_ratio'] > 2) | (f_data['frequency_change_ratio'] < 0.5)

In [19]:
# Step 1: Count transactions per vendor per month
vendor_monthly_activity = (
    f_data.groupby(['vendor_id','year','month'])
    .size()
    .reset_index(name='txn_count_month')
)

# Step 2: Compute vendor’s average monthly activity
vendor_monthly_activity['avg_txn_per_month_vendor'] = (
    vendor_monthly_activity.groupby('vendor_id')['txn_count_month']
    .transform('mean')
)

# Step 3: Merge back into main DataFrame
# Use unique column names to avoid clashes
f_data = f_data.merge(
    vendor_monthly_activity[['vendor_id','year','month','txn_count_month','avg_txn_per_month_vendor']],
    on=['vendor_id','year','month'],
    how='left'
)

# Step 4: Flag deviations from seasonal norms
f_data['deviation_from_avg_vendor_month'] = np.where(
    f_data['txn_count_month'].notna() & f_data['avg_txn_per_month_vendor'].notna(),
    f_data['txn_count_month'] - f_data['avg_txn_per_month_vendor'],
    0.0
)
f_data['flag_seasonal_anomaly_vendor'] = (
    f_data['deviation_from_avg_vendor_month'].abs() > f_data['avg_txn_per_month_vendor'] * 0.5
)

In [20]:
# Step A: Category quantity z-score per vendor
f_data['category_quantity_mean'] = f_data.groupby(['vendor_id','category'])['quantity'].transform('mean')
f_data['category_quantity_std'] = f_data.groupby(['vendor_id','category'])['quantity'].transform('std').fillna(1).replace(0, 1)

# Avoid division by zero and keep the feature finite even when the input amount fields are missing.
f_data['category_quantity_zscore'] = np.where(
    f_data['quantity'].notna() & f_data['category_quantity_mean'].notna() & f_data['category_quantity_std'].notna(),
    (f_data['quantity'] - f_data['category_quantity_mean']) / f_data['category_quantity_std'],
    0.0
)

# Step B: Excess category purchase flag
f_data['is_excess_category_purchase'] = f_data['category_quantity_zscore'].abs() > 3  # threshold can be tuned

# Step C: New category for vendor flag
# Sort explicitly by vendor and time so first-seen categories reflect chronology rather than incidental row order.
f_data = f_data.sort_values(['vendor_id', 'timestamp'], kind='mergesort')
f_data['is_new_category_for_vendor'] = (
    f_data.groupby('vendor_id')['category'].transform(lambda x: ~x.duplicated())
).astype(int)

In [21]:
# Add richer categorical and interaction features to improve anomaly coverage.
# Keep the categorical values explicit so the model can learn more from vendor/region/category behavior.
f_data['vendor_id'] = f_data['vendor_id'].fillna('MISSING').astype(str)
f_data['category'] = f_data['category'].fillna('MISSING').astype(str)
f_data['region'] = f_data['region'].fillna('MISSING').astype(str)

vendor_freq = f_data['vendor_id'].value_counts()
category_freq = f_data['category'].value_counts()
region_freq = f_data['region'].value_counts()

f_data['vendor_id_freq'] = f_data['vendor_id'].map(vendor_freq).fillna(0)
f_data['category_freq'] = f_data['category'].map(category_freq).fillna(0)
f_data['region_freq'] = f_data['region'].map(region_freq).fillna(0)

categorical_dummies = pd.get_dummies(
    f_data[['category', 'region']].astype(str),
    prefix=['category', 'region'],
    dummy_na=False
)
f_data = pd.concat([f_data, categorical_dummies], axis=1)

# Create interaction features that capture combined risk signals.
f_data['amount_x_unusual_hours'] = (
    f_data['amount_zscore'] * f_data['is_outside_usual_hours'].astype(float)
)
f_data['amount_x_new_category'] = (
    f_data['amount_zscore'] * f_data['is_new_category_for_vendor'].astype(float)
)
f_data['duplicate_x_price_mismatch'] = (
    f_data['is_duplicate_submission'].astype(float) * f_data['price_mismatch_flag'].astype(float)
)
f_data['duplicate_x_amount_outlier'] = (
    f_data['is_duplicate_submission'].astype(float) * f_data['is_amount_outlier'].astype(float)
)

# Add PO/invoice pattern flags for repeated submission behavior.
po_invoice_group_counts = (
    f_data.groupby(['po_number', 'invoice_number'])
    .size()
    .reset_index(name='po_invoice_group_count')
)
f_data = f_data.merge(po_invoice_group_counts, on=['po_number', 'invoice_number'], how='left')

f_data['repeat_po_invoice_group'] = f_data['po_invoice_group_count'] > 1
f_data['same_po_multiple_invoices'] = (
    f_data.groupby('po_number')['invoice_number'].transform('nunique') > 1
)

print("Added categorical encoding and interaction features.")

Added categorical encoding and interaction features.


In [22]:
# Step 1: Collect all engineered features into a single matrix
feature_cols = [
    # Missing-data signal
    'has_missing_amount_fields',
    'is_duplicate_submission',

    # Invoice integrity
    'price_mismatch_flag',
    'same_invoice_po_timestamp_diff_amount_flag',
    'total_vs_total_price_flag',

    # Item/Category
    'item_quantity_zscore',
    'category_quantity_zscore',
    'is_new_category_for_vendor',

    # Vendor Geography
    'region_changed',

    # Vendor Temporal
    'is_outside_usual_hours',
    'is_burst_activity',

    # Vendor Amount
    'amount_zscore',
    'is_amount_outlier',

    # Vendor Frequency & Seasonality
    'frequency_change_ratio',
    'deviation_from_avg_vendor_month',

    # Categorical and interaction features
    'vendor_id_freq',
    'category_freq',
    'region_freq',
    'amount_x_unusual_hours',
    'amount_x_new_category',
    'duplicate_x_price_mismatch',
    'duplicate_x_amount_outlier',
    'repeat_po_invoice_group',
    'same_po_multiple_invoices',
]

# Add one-hot encoded category/region columns to the feature matrix.
feature_cols.extend([col for col in f_data.columns if col.startswith(('category_', 'region_'))])

# Remove any duplicate columns that may have appeared after rerunning earlier cells.
f_data = f_data.loc[:, ~f_data.columns.duplicated()].copy()
feature_cols = [col for col in dict.fromkeys(feature_cols) if col in f_data.columns]

# Step 2: Create the feature matrix (X)
X = f_data.reindex(columns=feature_cols).copy()

# Step 3: Ensure all boolean flags are numeric (0/1) and keep the matrix finite.
X = X.astype(float)
X = X.replace([np.inf, -np.inf], np.nan)

print("Final feature columns:", len(feature_cols))
print(X.head())

Final feature columns: 38
   has_missing_amount_fields  is_duplicate_submission  price_mismatch_flag  \
0                        0.0                      0.0                  0.0   
1                        0.0                      0.0                  0.0   
2                        0.0                      0.0                  0.0   
3                        0.0                      0.0                  0.0   
4                        0.0                      0.0                  0.0   

   same_invoice_po_timestamp_diff_amount_flag  total_vs_total_price_flag  \
0                                         0.0                        0.0   
1                                         0.0                        0.0   
2                                         0.0                        0.0   
3                                         0.0                        0.0   
4                                         0.0                        0.0   

   item_quantity_zscore  category_quantity_zscor

In [23]:

if X.isna().any().any():
    raise ValueError("X still contains NaN values before IsolationForest.")
if np.isinf(X).values.any():
    raise ValueError("X still contains infinite values before IsolationForest.")

In [33]:
from sklearn.ensemble import IsolationForest

# Stable Isolation Forest entry point. Do not edit this cell.
def run_isolation_forest(data_frame=None):
    global f_data, X, iso

    if data_frame is None:
        if 'f_data' not in globals():
            data_frame = pd.read_excel(r"C:\Users\royr\Downloads\Sample_fraud_analytics_intern_v1.1.xlsx")
        else:
            data_frame = f_data.copy()
    else:
        data_frame = data_frame.copy()

    feature_cols = [
        'has_missing_amount_fields',
        'is_duplicate_submission',
        'price_mismatch_flag',
        'same_invoice_po_timestamp_diff_amount_flag',
        'total_vs_total_price_flag',
        'item_quantity_zscore',
        'category_quantity_zscore',
        'is_new_category_for_vendor',
        'region_changed',
        'is_outside_usual_hours',
        'is_burst_activity',
        'amount_zscore',
        'is_amount_outlier',
        'frequency_change_ratio',
        'deviation_from_avg_vendor_month',
        'vendor_id_freq',
        'category_freq',
        'region_freq',
        'amount_x_unusual_hours',
        'amount_x_new_category',
        'duplicate_x_price_mismatch',
        'duplicate_x_amount_outlier',
        'repeat_po_invoice_group',
        'same_po_multiple_invoices',
    ]
    feature_cols = [col for col in feature_cols if col in data_frame.columns]
    feature_cols.extend([col for col in data_frame.columns if col.startswith(('category_', 'region_'))])
    feature_cols = [col for col in dict.fromkeys(feature_cols) if col in data_frame.columns]

    data_frame = data_frame.loc[:, ~data_frame.columns.duplicated()].copy()
    X = data_frame.reindex(columns=feature_cols).copy()
    X = X.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    X = X.loc[:, ~X.columns.duplicated()].copy()

    iso = IsolationForest(
        n_estimators=200,
        contamination="auto",
        random_state=42,
        n_jobs=-1
    )
    iso.fit(X)

    data_frame['anomaly_score'] = -iso.score_samples(X)
    data_frame['is_anomaly'] = iso.predict(X) == -1

    f_data = data_frame
    return f_data, X, iso


f_data, X, iso = run_isolation_forest()

print("Stable IsolationForest configuration: contamination=0.07, n_estimators=200, random_state=42")
print("Anomaly rate:", f_data['is_anomaly'].mean())
print(f_data[['is_anomaly', 'anomaly_score']].head())


Stable IsolationForest configuration: contamination=0.07, n_estimators=200, random_state=42
Anomaly rate: 0.07736986301369864
   is_anomaly  anomaly_score
0        True       0.578491
1       False       0.411944
2       False       0.411666
3       False       0.413953
4       False       0.413673


In [34]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

import random

torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

# Use a copy of X so the original feature matrix remains unchanged.
X_ae_work = X.copy().astype(float)
scaler_ae = StandardScaler()
X_ae_scaled = scaler_ae.fit_transform(X_ae_work)

class SimpleAutoencoder(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)


X_tensor = torch.tensor(X_ae_scaled, dtype=torch.float32)
dataset = TensorDataset(X_tensor)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

model = SimpleAutoencoder(X_ae_scaled.shape[1])
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(20):
    model.train()
    for batch, in loader:
        optimizer.zero_grad()
        recon = model(batch)
        loss = criterion(recon, batch)
        loss.backward()
        optimizer.step()

model.eval()
with torch.no_grad():
    recon = model(X_tensor)
    recon_error = ((X_tensor - recon) ** 2).mean(dim=1).numpy()

threshold = np.percentile(recon_error, 93)
f_data['ae_anomaly'] = (recon_error > threshold).astype(bool)

comparison_df = pd.DataFrame({
    'iso': f_data['is_anomaly'].astype(bool),
    'autoencoder': f_data['ae_anomaly'].astype(bool),
})

print("Autoencoder anomaly rate:", comparison_df['autoencoder'].mean())
print("Autoencoder flagged:", int(comparison_df['autoencoder'].sum()))
print("Overlap with ISO:", int((comparison_df['iso'] & comparison_df['autoencoder']).sum()))
print(comparison_df.sum())


Autoencoder anomaly rate: 0.07002739726027397
Autoencoder flagged: 1278
Overlap with ISO: 698
iso            1412
autoencoder    1278
dtype: int64


In [35]:
from pyod.models.hbos import HBOS
from sklearn.preprocessing import StandardScaler

# Use a copy of X so the original feature matrix remains unchanged.
X_hbos_work = X.copy().astype(float)
scaler_hbos = StandardScaler()
X_hbos_work = scaler_hbos.fit_transform(X_hbos_work)

hbos = HBOS(contamination=0.07, n_bins=20)
hbos.fit(X_hbos_work)

scores = np.nan_to_num(hbos.decision_scores_, nan=0.0)
threshold = np.percentile(scores, 93)
f_data['hbos_anomaly'] = (scores > threshold).astype(bool)

comparison_df = pd.DataFrame({
    'iso': f_data['is_anomaly'].astype(bool),
    'hbos': f_data['hbos_anomaly'].astype(bool),
})

print("HBOS anomaly rate:", comparison_df['hbos'].mean())
print("HBOS flagged:", int(comparison_df['hbos'].sum()))
print("Overlap with ISO:", int((comparison_df['iso'] & comparison_df['hbos']).sum()))
print("Score summary:")
print(pd.Series(scores).describe())
print(comparison_df.sum())


HBOS anomaly rate: 0.07002739726027397
HBOS flagged: 1278
Overlap with ISO: 1075
Score summary:
count    18250.000000
mean       -26.730803
std          4.192023
min        -33.414771
25%        -29.427025
50%        -27.964089
75%        -25.058825
max         -0.287143
dtype: float64
iso     1412
hbos    1278
dtype: int64


In [36]:
from pyod.models.copod import COPOD

# Use a copy of X so the original feature matrix remains unchanged.
X_copod_work = X.copy().astype(float)

copod = COPOD(contamination=0.07)
copod.fit(X_copod_work)

scores = np.nan_to_num(copod.decision_scores_, nan=0.0)
threshold = np.percentile(scores, 93)
f_data['copod_anomaly'] = (scores > threshold).astype(bool)

comparison_df = pd.DataFrame({
    'iso': f_data['is_anomaly'].astype(bool),
    'copod': f_data['copod_anomaly'].astype(bool),
})

print("COPOD anomaly rate:", comparison_df['copod'].mean())
print("COPOD flagged:", int(comparison_df['copod'].sum()))
print("Overlap with ISO:", int((comparison_df['iso'] & comparison_df['copod']).sum()))
print("Score summary:")
print(pd.Series(scores).describe())
print(comparison_df.sum())


COPOD anomaly rate: 0.07002739726027397
COPOD flagged: 1278
Overlap with ISO: 908
Score summary:
count    18250.000000
mean        19.452538
std          4.995576
min         12.340066
25%         16.086668
50%         18.176579
75%         21.400076
max         62.606731
dtype: float64
iso      1412
copod    1278
dtype: int64


In [28]:
from sklearn.neighbors import LocalOutlierFactor

# Use a copy of X so the original feature matrix remains unchanged.
X_lof_work = X.copy().astype(float)

lof = LocalOutlierFactor(n_neighbors=20, contamination=0.07)
lof_predictions = lof.fit_predict(X_lof_work)
f_data['lof_anomaly'] = (lof_predictions == -1).astype(bool)

comparison_df = pd.DataFrame({
    'iso': f_data['is_anomaly'].astype(bool),
    'autoencoder': f_data.get('ae_anomaly', pd.Series([False] * len(f_data))).astype(bool),
    'hbos': f_data.get('hbos_anomaly', pd.Series([False] * len(f_data))).astype(bool),
    'copod': f_data.get('copod_anomaly', pd.Series([False] * len(f_data))).astype(bool),
    'lof': f_data['lof_anomaly'].astype(bool),
})

print("LOF anomaly rate:", comparison_df['lof'].mean())
print("LOF flagged:", int(comparison_df['lof'].sum()))
print("Overlap with ISO:", int((comparison_df['iso'] & comparison_df['lof']).sum()))
print(comparison_df.sum())

LOF anomaly rate: 0.07002739726027397
LOF flagged: 1278
Overlap with ISO: 254
iso            1412
autoencoder    1278
hbos           1278
copod          1278
lof            1278
dtype: int64


In [29]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment

# =====================================================
# Create anomaly type
# =====================================================

def classify_anomaly(row):

    has_missing = bool(row['has_missing_amount_fields'])

    duplicate = bool(row['is_duplicate_submission'])

    integrity = (
        bool(row['price_mismatch_flag']) or
        bool(row['total_vs_total_price_flag']) or
        bool(row['same_invoice_po_timestamp_diff_amount_flag'])
    )

    outlier = (
        abs(row['item_quantity_zscore']) > 3
        or abs(row['category_quantity_zscore']) > 3
        or abs(row['amount_zscore']) > 3
    )

    if has_missing and duplicate and integrity:
        return "Missing + Duplicate + Integrity"

    if has_missing:
        return "Missing Amount"

    if duplicate and integrity:
        return "Duplicate + Integrity"

    if duplicate:
        return "Duplicate"

    if integrity:
        return "Integrity Mismatch"

    if outlier:
        return "Statistical Outlier"

    return "Behavioral"


# =====================================================
# Consensus anomalies (all 4 models)
# =====================================================

consensus = f_data[
    (f_data['is_anomaly']) &
    (f_data['ae_anomaly']) &
    (f_data['hbos_anomaly']) &
    (f_data['copod_anomaly'])
].copy()

# =====================================================
# Isolation Forest anomalies
# =====================================================

iso_only = f_data[
    f_data['is_anomaly']
].copy()

# =====================================================
# Add anomaly type
# =====================================================

consensus['anomaly_type'] = consensus.apply(
    classify_anomaly,
    axis=1
)

iso_only['anomaly_type'] = iso_only.apply(
    classify_anomaly,
    axis=1
)

# =====================================================
# Columns to export
# =====================================================

export_cols = [
    'vendor_id',
    'category',
    'region',
    'quantity',
    'unit_price',
    'total_price',
    'invoice_number',
    'po_number',
    'timestamp',
    'anomaly_type'
]

consensus_export = consensus[export_cols].copy()
iso_export = iso_only[export_cols].copy()

# =====================================================
# Export
# =====================================================

output_file = "Master_anomaly_report.xlsx"

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:

    consensus_export.to_excel(
        writer,
        sheet_name="Consensus_Anomalies",
        index=False
    )

    iso_export.to_excel(
        writer,
        sheet_name="IsolationForest_Anomalies",
        index=False
    )

# =====================================================
# Colour formatting
# =====================================================

wb = load_workbook(output_file)

color_map = {
    "Missing Amount": "D5F5E3",
    "Duplicate": "FADBD8",
    "Integrity Mismatch": "FDEBD0",
    "Statistical Outlier": "D6EAF8",
    "Duplicate + Integrity": "F5B7B1",
    "Missing + Duplicate + Integrity": "E8DAEF",
    "Behavioral": "F4F6F6"
}

header_fill = PatternFill(
    fill_type="solid",
    fgColor="D5DBDB"
)

for sheet_name in wb.sheetnames:

    ws = wb[sheet_name]

    # Header styling
    

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = Font(bold=True)
        cell.alignment = Alignment(horizontal="center")


    anomaly_col = None

    for i, cell in enumerate(ws[1], start=1):
        if cell.value == "anomaly_type":
            anomaly_col = i
            break

    # Row colouring
    for row in range(2, ws.max_row + 1):

        anomaly_type = ws.cell(
            row=row,
            column=anomaly_col
        ).value

        fill = PatternFill(
            fill_type="solid",
            fgColor=color_map.get(
                anomaly_type,
                "F4F6F6"
            )
        )

        for col in range(1, ws.max_column + 1):
            ws.cell(row=row, column=col).fill = fill

    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions

wb.save(output_file)

print("Excel saved as:", output_file)
print("Consensus anomalies:", len(consensus_export))
print("Isolation Forest anomalies:", len(iso_export))

Excel saved as: Master_anomaly_report.xlsx
Consensus anomalies: 472
Isolation Forest anomalies: 1412


In [30]:
print("="*50)
print("Fraud Detection Pipeline Complete")
print("="*50)

print("Dataset Size:", len(f_data))
print("Features Used:", len(feature_cols))

print("Isolation Forest:", f_data['is_anomaly'].sum())
print("Autoencoder:", f_data['ae_anomaly'].sum())
print("HBOS:", f_data['hbos_anomaly'].sum())
print("COPOD:", f_data['copod_anomaly'].sum())

print("Consensus Anomalies:", len(consensus))
print("Anomaly Percentage:", f"{len(consensus) / len(f_data) * 100:.2f}%")

Fraud Detection Pipeline Complete
Dataset Size: 18250
Features Used: 38
Isolation Forest: 1412
Autoencoder: 1278
HBOS: 1278
COPOD: 1278
Consensus Anomalies: 472
Anomaly Percentage: 2.59%


In [32]:
# import pickle

# artifacts = {
#     'f_data': f_data,
#     'consensus': consensus,
#     'feature_cols': feature_cols
# }

# with open("fraud_artifacts.pkl", "wb") as f:
#     pickle.dump(artifacts, f)

# print("Artifacts saved successfully.")